In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
tb_data = pd.read_csv("kncv_nigeria_tb_data_raw (1).csv")
print("Raw shape:", tb_data.shape)


Raw shape: (60150, 26)


In [15]:
# Age is self-reported — impossible entries (999, 0) replaced with null
tb_data['age'] = tb_data['age'].replace({999: np.nan, 0: np.nan})
print("Age nulls after fix:", tb_data['age'].isna().sum())

Age nulls after fix: 1150


In [16]:
# ~150 duplicate patient_ids were planted to simulate double data entry
before = tb_data.shape[0]
tb_data = tb_data.drop_duplicates(subset='patient_id', keep='first').reset_index(drop=True)
after = tb_data.shape[0]
print(f"Duplicates removed: {before - after}")

Duplicates removed: 150


In [17]:
date_cols = ['screening_date', 'diagnosis_date', 'treatment_start_date', 'outcome_date']
for col in date_cols:
    tb_data[col] = pd.to_datetime(tb_data[col], errors='coerce')
print("Screening date nulls:", tb_data['screening_date'].isna().sum())


Screening date nulls: 0


In [18]:
def standardize_referred_by(value):
    if pd.isna(value):
        return np.nan
    value = value.strip().lower()
    if any(x in value for x in ['comm', 'cv']):
        return 'Community Volunteer'
    elif any(x in value for x in ['health', 'hw', 'h/w', 'hlth']):
        return 'Health Worker'
    elif any(x in value for x in ['self', 'walk']):
        return 'Self Referral'
    elif any(x in value for x in ['contact', 'ct', 'cont']):
        return 'Contact Tracing'
    elif any(x in value for x in ['plhiv', 'hiv', 'art']):
        return 'PLHIV Clinic'
    else:
        return value

tb_data['referred_by'] = tb_data['referred_by'].apply(standardize_referred_by)

# Fix stragglers
tb_data['referred_by'] = tb_data['referred_by'].replace({
    'htlh worker': 'Health Worker',
    'comunity volunteer': 'Community Volunteer'
})
print("referred_by categories:", tb_data['referred_by'].value_counts().to_dict())

referred_by categories: {'Community Volunteer': 12138, 'Self Referral': 12034, 'Contact Tracing': 11959, 'PLHIV Clinic': 11941, 'Health Worker': 11928}


In [19]:
# ── STEP 5: STANDARDIZE LTFU REASON ──────────────────────────────
def standardize_ltfu(value):
    if pd.isna(value):
        return np.nan
    value = value.strip().lower()
    if any(x in value for x in ['reloc', 'moved', 'move', 'village']):
        return 'Relocated'
    elif any(x in value for x in ['refus']):
        return 'Refused Treatment'
    elif any(x in value for x in ['unknown', "don't know", 'not known', 'nk']):
        return 'Unknown'
    elif any(x in value for x in ['financ', 'money', 'afford']):
        return 'Financial Constraints'
    elif any(x in value for x in ['better', 'fine']):
        return 'Felt Better'
    elif any(x in value for x in ['side', 'adverse', 'reaction']):
        return 'Side Effects'
    else:
        return value

tb_data['lost_to_followup_reason'] = tb_data['lost_to_followup_reason'].apply(standardize_ltfu)
print("LTFU categories:", tb_data['lost_to_followup_reason'].value_counts().to_dict())


LTFU categories: {'Relocated': 928, 'Refused Treatment': 802, 'Felt Better': 782, 'Side Effects': 731, 'Financial Constraints': 727, 'Unknown': 716}


In [20]:
# ── STEP 6: DERIVE TREATMENT DURATION ────────────────────────────
# Not stored in raw data — calculated from dates (more realistic)
tb_data['treatment_duration_months'] = (
    (tb_data['outcome_date'] - tb_data['treatment_start_date']).dt.days / 30
).round(1)

# Null out negative durations (date inconsistencies)
negative_count = (tb_data['treatment_duration_months'] < 0).sum()
tb_data.loc[tb_data['treatment_duration_months'] < 0, 'treatment_duration_months'] = np.nan
print(f"Negative durations nulled: {negative_count}")

# ── FINAL CHECK ───────────────────────────────────────────────────
print("\nFinal shape:", tb_data.shape)
print("\nNull counts:\n", tb_data.isnull().sum())
print("\nData types:\n", tb_data.dtypes)

Negative durations nulled: 0

Final shape: (60000, 27)

Null counts:
 patient_id                       0
state                            0
LGA                              0
sex                              0
age                           1150
screening_date                   0
case_finding_method              0
referred_by                      0
presumptive_tb                   0
is_plhiv                         0
is_child_under15                 0
is_contact_of_tb_patient         0
is_high_risk_setting             0
diagnosis_date               11964
diagnostic_tool              11964
specimen_type                11964
tb_type                      24116
treatment_start_date         24116
treatment_regimen            24116
dots_method                  24116
treatment_facility_level     24116
treatment_outcome            24116
outcome_date                 24116
lost_to_followup_reason      55314
tpt_eligible                 35884
tpt_initiated                35884
treatment_duration_m

In [26]:
# Save cleaned dataset
tb_data.to_csv('kncv_nigeria_tb_data_cleaned.csv', index=False)
print("Cleaned dataset saved as 'kncv_nigeria_tb_data_cleaned.csv'")

Cleaned dataset saved as 'kncv_nigeria_tb_data_cleaned.csv'
